# Assignment · Weekend support desk


An online retailer is running a weekend sale. At shift handover, the support lead needs a short update on tickets still waiting. At the same time, the checkout team wants to tell a new customer how long support might take and flag likely long waits for human review. The review team has capacity for **14 alerts in a batch of 36 tickets**.

Build a handover report and answer two new customer cases using the methods from [APIs and your first ML model](01_apis_and_first_ml_model.ipynb). Budget around 90 minutes. Replace the `TODO` comments with code and fill in the short written answers.

### Set up

Use a Python 3.10 or newer Jupyter kernel. If imports fail, run `%pip install numpy pandas scikit-learn` in a separate notebook cell and restart the kernel. Run Jupyter from the course repository root or from the `04-machine-learning-foundations` folder. The assignment uses sample API responses and runs offline.

### Read the two datasets

Both files contain synthetic practice data. They represent two different questions about the same support desk.

| File | What one row represents | Fields used |
| --- | --- | --- |
| [Shift snapshot](data/weekend_support_snapshot.csv) | One ticket in the current handover batch | `ticket_id`, `category`, `status`, `elapsed_wait_minutes` |
| [Wait history](data/weekend_support_wait_history.csv) | One completed ticket from a previous shift | `ticket_id`, `queue_size` at arrival, `actual_wait_minutes` recorded at pickup |

For the handover, consider elapsed waits for **open** tickets only. A blank wait means unknown and zero is a known wait. For the models, use the separate history and predict from **queue size at arrival**. Ticket IDs identify rows and are excluded from the model features.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

# This works when the kernel starts in the repository root or module folder.
module = Path("04-machine-learning-foundations")
if not (module / "data").is_dir():
    module = Path(".")

data_folder = module / "data"
snapshot = pd.read_csv(data_folder / "weekend_support_snapshot.csv")
wait_history = pd.read_csv(data_folder / "weekend_support_wait_history.csv")

display(snapshot.head())
display(wait_history.head())
print(f"Snapshot rows: {len(snapshot)}")
print(f"History rows: {len(wait_history)}")

,ticket_id,category,status,elapsed_wait_minutes
0,301,delivery,open,0.0
1,302,returns,open,12.0
2,303,payments,open,18.0
3,304,delivery,open,29.0
4,305,returns,open,30.0


,ticket_id,queue_size,actual_wait_minutes
0,1001,20,49.3
1,1002,12,27.9
2,1003,19,32.1
3,1004,7,23.9
4,1005,0,7.0


Snapshot rows: 14
History rows: 180


## 1. Calculate the handover facts · 20 points

Write `calculate_shift_facts(data, threshold_minutes=30)` and return a dictionary with the following fields.

| Key | What to calculate |
| --- | --- |
| `total_tickets` | All rows in the snapshot |
| `open_tickets` | Rows with status `open` |
| `unknown_open_waits` | Open rows with a blank elapsed wait |
| `open_by_category` | Dictionary with counts of all open rows in each category |
| `threshold_minutes` | The threshold passed to your function |
| `open_at_or_above_threshold` | Open rows with a known wait **at or above** the threshold |
| `average_known_open_wait_minutes` | Mean of known open waits, or `None` when no open waits are known |

Run the function at 30 and 45 minutes. Explain how you handled the ticket at exactly 30, the open ticket with zero wait, and the closed ticket with a 72-minute wait. Use ordinary Python numbers in the dictionary so it can be written as JSON.

**Checkpoint.** The snapshot has 14 tickets and 10 are open. Two of the open tickets have unknown waits. Changing the threshold should change the threshold field and the count of tickets meeting it.

In [ ]:
# TODO: Define calculate_shift_facts(data, threshold_minutes=30).
# Filter the open rows, then separate the known waits for the mean and threshold.
# Save the two results as facts_at_30 and facts_at_45 and display both.

**Your explanation.** _Write two or three sentences explaining how the 30-minute, zero-minute and closed tickets were handled._

## 2. Build and save the handover report · 20 points

1. Build a prompt for a two-sentence shift update. Use `json.dumps(facts_at_30)` to include the verified facts. Ask the model to preserve the difference between “at least 30 minutes” and “over 30 minutes.”
2. Use the starter responses to practice parsing. Extract the text from the example Ollama response and the example Groq response and print each. Explain in one sentence what a live HTTP status of `200` tells you and what you still need to verify in the returned text.
3. Write a two-sentence handover using your calculated facts and put it in a response dictionary with either the Groq or Ollama shape. Read it with your parser. Build a report dictionary with `facts`, `summary` and `summary_source`. For the sample response, use `example_groq` or `example_ollama` as the source.
4. Test the incomplete Groq response as a second case. The report should keep the same facts, set `summary_source` to `fallback`, and use a short handover constructed from those facts. Show which field or list position was missing.
5. Save your successful example report as `weekend_support_report.json`, then reload it and check that the facts and source match. Use the output path from the starter cell.

**Optional live call.** If your provider is set up, you can use a live reply for step 3 and label the source `groq` or `ollama`. Follow the lesson's provider setup and `.env` instructions. Compare the returned wording with the facts before saving. The sample-response path earns full credit.

In [ ]:
# Two common JSON response shapes and one incomplete response to test.
example_ollama_reply = {
    "message": {"role": "assistant", "content": "This is a sample Ollama reply."}
}
example_groq_reply = {
    "choices": [
        {"message": {"role": "assistant", "content": "This is a sample Groq reply."}}
    ]
}
incomplete_groq_reply = {"choices": []}

output_folder = module / "weekend_support_outputs"
report_path = output_folder / "weekend_support_report.json"

In [ ]:
# TODO: Build the prompt and extract text from the two response shapes.
# TODO: Prepare your own two-sentence example, then build both report cases.
# TODO: Create output_folder and write the successful report to report_path.
# Reload the file with json.load() and check the three keys and saved facts.

**Your explanation.** _Explain what HTTP 200 confirms and what a human should check in the handover._

## 3. Estimate a customer's wait · 25 points

Use `wait_history` to fit and compare two approaches.

1. Create a one-column feature DataFrame `X` with `queue_size`, and a target Series `y` from `actual_wait_minutes`. Use `train_test_split` with `test_size=0.20` and `random_state=42` to set aside the test set. Then split the remaining rows with `test_size=0.25` and the same seed. Print the three row counts to check your split.
2. Define a baseline that predicts the **mean training wait** for every ticket. Fit `LinearRegression` on the training features and actual waits.
3. Score both approaches on the **same validation rows**. Display a two-row table with **MAE in minutes**, **RMSE in minutes**, and **R²**. Round for display after scoring.
4. Choose the approach with the lowest validation MAE. If they tie, choose the baseline. Explain why the lead may prefer to quote MAE and when they might look at RMSE too.

**Checkpoint.** You should have **108 training**, **36 validation** and **36 test** rows. Keep the test set for Part 5.

In [ ]:
# TODO: Build X and y, then save the train, validation and test splits.
# TODO: Fit LinearRegression and make baseline and model validation predictions.
# TODO: Display the two-row validation metrics table and record your choice.

**Your recommendation.** _Which approach would you use for the wait estimate, and what does its validation MAE mean in minutes? Why might the team also monitor RMSE?_

## 4. Decide when to raise a long-wait alert · 25 points

A positive case is a ticket that **actually waited more than 30 minutes**. This is a separate definition from the handover count of current tickets waiting at least 30 minutes.

1. Use the **same split** as Part 3. Convert the actual waits to 0/1 labels using `actual_wait_minutes > 30`. Fit `LogisticRegression` on the training features and training labels.
2. Use `predict_proba(X_val)` to get a score for the positive class. Use `model.classes_` to find which column belongs to label 1.
3. Try cutoffs **0.30**, **0.50** and **0.70** on the validation scores. Raise an alert when the score is **at least** the cutoff. Display one row per cutoff with alerts sent, false alerts (FP), missed long waits (FN), precision, recall and F1. Pass `labels=[0, 1]` to the confusion matrix so the four boxes are in a known order.
4. Choose the cutoff with the **highest recall** among rows that send **14 or fewer alerts**. If recall ties, choose higher precision. If both tie, choose the cutoff that sends fewer alerts. Show the chosen confusion matrix and explain the choice.

**Think about the decision.** Why might a cutoff with better recall be impractical? Why can high accuracy hide missed long waits?

In [ ]:
# TODO: Fit the classifier on 0/1 training labels and get validation scores.
# TODO: Compare all three cutoffs and display the selected confusion matrix.
# Record the selected cutoff before opening the test results in Part 5.

**Your recommendation.** _State which cutoff you chose, how many reviews it sends, how many real long waits are missed, and why you chose it. Explain how accuracy alone could hide missed long waits._

## 5. Check the final choices and answer two customers · 10 points

1. Apply the selected wait approach to the **test rows**. Report test MAE and RMSE. Apply the selected alert cutoff to the classifier's test scores. Report test precision, recall, F1 and number of alerts. State whether test review volume still fits the 14-alert budget. Keep your validation choices fixed.
2. A customer arrives with **5** tickets ahead. Another arrives with **19** tickets ahead. Create a two-row DataFrame using the same feature column as in training. For each ticket, display the estimated wait in minutes, probability of a long wait and whether the selected cutoff triggers an alert.
3. Write one short customer-facing message per ticket. Call the wait a prediction, and explain the review action when the policy triggers it.

In [ ]:
incoming_tickets = pd.DataFrame({"queue_size": [5, 19]})

# TODO: Score the frozen models and policy on the test set.
# TODO: Add wait estimates, long-wait scores and alert decisions to incoming_tickets.

**Customer with 5 tickets ahead.** _Write your response._

**Customer with 19 tickets ahead.** _Write your response._

### Before submitting

Submit this completed notebook with saved outputs and your `weekend_support_report.json`.

- Restart the kernel and run all cells. The datasets should load with their relative paths.
- Confirm the handover report matches the counts at the 30-minute threshold and that the incomplete reply can use the same facts.
- Show your validation comparison tables and which choices they led to. Report the test results after those choices.
- Include your explanations and both customer messages. Keep any API key in your private `.env`.